# 06-Baseline y Evaluación

## Objetivo

Entrenar un modelo simple sobre las caracteristicas extraidas con OpenCV para obtener el baseline contra el cual se compararan con el modelo.

## 1. Importacion y carga de train

In [9]:
import sys
sys.path.append("..")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

PROCESSED = Path("../data/processed")
feat_train = pd.read_csv(PROCESSED / "features_train.csv")

cols = [c for c in feat_train.columns if c not in ("image_id", "etiqueta")]
X_train = feat_train[cols]
y_train = feat_train["etiqueta"]

print(X_train.shape, y_train.value_counts().to_dict())
print("Valores nulos:", X_train.isna().sum().sum())

(982, 18) {1: 491, 0: 491}
Valores nulos: 0


## 2. Comparacion con validacion cruzada

Pruebo 3 modelos. El Dummy siempre predice la misma clase y marca el piso.

In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modelos = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Regresión logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
}

scoring = {"recall": "recall", "f1_macro": "f1_macro", "accuracy": "accuracy"}

filas = []
for nombre, modelo in modelos.items():
    r = cross_validate(modelo, X_train, y_train, cv=cv, scoring=scoring)
    filas.append({
        "modelo": nombre,
        "recall_mel": r["test_recall"].mean(),
        "recall_std": r["test_recall"].std(),
        "f1_macro": r["test_f1_macro"].mean(),
        "f1_std": r["test_f1_macro"].std(),
        "accuracy": r["test_accuracy"].mean(),
    })

resultados = pd.DataFrame(filas).set_index("modelo").round(3)
resultados

,recall_mel,recall_std,f1_macro,f1_std,accuracy
modelo,,,,,
Dummy,0.200,0.400,0.333,0.001,0.499
Regresión logística,0.825,0.048,0.797,0.008,0.797
Random Forest,0.864,0.045,0.819,0.013,0.820


## 3. Eleccion y entrenamiento final

In [ ]:
modelo_final = RandomForestClassifier(n_estimators=300, random_state=42)
modelo_final.fit(X_train, y_train)



lab2_media      0.225
solidez         0.084
lab3_desvio     0.081
hsv2_media      0.078
hsv3_media      0.063
hsv2_desvio     0.059
circularidad    0.054
lab3_media      0.042
lab2_desvio     0.042
asimetria       0.039
dtype: float64


## 4. Evaluacion

In [12]:
from sklearn.metrics import recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

feat_test = pd.read_csv(PROCESSED / "features_test.csv")
X_test = feat_test[cols]
y_test = feat_test["etiqueta"]

pred = modelo_final.predict(X_test)

print("Recall melanoma:", round(recall_score(y_test, pred), 3))
print("F1 macro:", round(f1_score(y_test, pred, average="macro"), 3))


Recall melanoma: 0.927
F1 macro: 0.869
